# Tasks for laboratory assignment 1

In [7]:
# imports section

import json
import requests

import pandas as pd
import matplotlib.pyplot as plt

from pprint import pprint
from bs4 import BeautifulSoup

## Extract webpage data given the url

Create a Python script that performs basic web scraping on a page to extract all the information into text and returns it as a string.
String should not contain tags.

In [8]:
def parse_web_page(url: str, *args, **kwargs) -> str:
    response = requests.get(url, *args, **kwargs)
    response.raise_for_status()  # Raise HTTPError if request failed

    # Parse HTML content
    soup = BeautifulSoup(response.text, "html.parser")

    # Extract only text, stripping scripts/styles
    for script in soup(["script", "style"]):
        script.extract()

    # Get clean text
    text = soup.get_text(separator="\n")

    # Clean up multiple newlines/whitespace
    lines = [line.strip() for line in text.splitlines()]
    text = "\n".join(filter(None, lines))

    return text


print(parse_web_page("https://fmi.chnu.edu.ua/")[:255])


# To parse a Wikipedia page, we need to specify the request headers, otherwise it will throw 403 error

# headers = {
#     "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
#     "accept-language": "uk-UA,uk;q=0.9,en-US;q=0.8,en;q=0.7",
#     "cache-control": "max-age=0",
#     "priority": "u=0, i",
#     "sec-ch-ua": '"Chromium";v="140", "Not=A?Brand";v="24", "Google Chrome";v="140"',
#     "sec-ch-ua-mobile": "?0",
#     "sec-ch-ua-platform": '"Windows"',
#     "sec-fetch-dest": "document",
#     "sec-fetch-mode": "navigate",
#     "sec-fetch-site": "none",
#     "sec-fetch-user": "?1",
#     "upgrade-insecure-requests": "1",
#     "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36",
# }

# print(
#     parse_web_page("https://en.wikipedia.org/wiki/Web_scraping", headers=headers)[:255]
# )

Головна - Факультет математики та інформатики
Перейти до основного вмісту
[email protected]
58012, Україна, м. Чернівці, вул. Університетська, 28
Новини
Всі
Загальні
Оголошення
Події
Студенту
Викладачу
Вітання
Діяльність
Наукова
Конференції
Семінари
Аспір


## Get data from the API

Create a python script that performs basic request to API endpoint and saves that data to a JSON file `result.json`.

In [9]:
def parse_api(api_url: str) -> None:
    response = requests.get(api_url)
    response.raise_for_status()  # Raise HTTPError if request failed

    data = response.json()
    result_file = "output/result.json"

    with open(result_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

    print(f"Data saved to `{result_file}` from {api_url}")


parse_api("https://api.github.com/")
# parse_api("https://jsonplaceholder.typicode.com/todos/")

Data saved to `output/result.json` from https://api.github.com/


## Parse the json file

Parse the `weather.json` file and return weather data for a specific date, that is given as a parameter. Return the data as an array.

In [10]:
def parse_json(date: str) -> list:
    with open("resources/weather.json", "r", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.DataFrame(data["daily"])

    # Normalize date column and input
    df["date"] = pd.to_datetime(df["date"]).dt.date
    target_date = pd.to_datetime(date).date()

    # Filter for the specific date
    result = df[df["date"] == target_date]

    return result.to_dict(orient="records")


target_date = "2024-8-19"
pprint(parse_json(target_date))

[{'date': datetime.date(2024, 8, 19),
  'humidity': 70,
  'max_temperature': 30.0,
  'min_temperature': 21.0,
  'precipitation': 5.0,
  'weather_description': 'Light rain',
  'wind_speed': 10.0}]


## Parse the csv file

Parse the `weather.csv` file and return weather data for a specific date, that is given as a parameter. Return the data as an array.

In [11]:
def parse_csv(date: str) -> list:
    # Load the CSV data
    df = pd.read_csv("resources/weather.csv")

    # Normalize date column and input
    df["CET"] = pd.to_datetime(df["CET"]).dt.date
    target_date = pd.to_datetime(date).date()

    # Filter for the specific date
    result = df[df["CET"] == target_date]

    return result.to_dict(orient="records")


target_date = "1997-5-22"
pprint(parse_csv(target_date))

[{'CET': datetime.date(1997, 5, 22),
  'CloudCover': 3.0,
  'Dew PointC': 11.0,
  'Events': nan,
  'Max Gust SpeedKm/h': nan,
  'Max Humidity': 88.0,
  'Max Sea Level PressurehPa': 1017,
  'Max TemperatureC': 25.0,
  'Max VisibilityKm': 10.0,
  'Max Wind SpeedKm/h': 11,
  'Mean Humidity': 54.0,
  'Mean Sea Level PressurehPa': 1015,
  'Mean TemperatureC': 18.0,
  'Mean VisibilityKm': 10.0,
  'Mean Wind SpeedKm/h': 3,
  'MeanDew PointC': 8.0,
  'Min DewpointC': 6.0,
  'Min Humidity': 34.0,
  'Min Sea Level PressurehPa': 1012,
  'Min TemperatureC': 10.0,
  'Min VisibilitykM': 10.0,
  'Precipitationmm': 0.0,
  'WindDirDegrees': 277}]


## Visualize data

Visualize the `weather.csv` data using matplotlib. Choose your own approach to data visualization. Save the results (as `.png`, `.webp` files etc., your choise) in this repository. 

In [12]:
def visualize_data() -> None:
    df = pd.read_csv("resources/weather.csv")

    # Convert date column to datetime
    df["CET"] = pd.to_datetime(df["CET"])

    # Set up the plotting style
    plt.style.use("default")
    fig_size = (12, 8)

    # 1. Time series of temperature over time - Yearly aggregation (Line plot - numeric x numeric sequential)
    plt.figure(figsize=fig_size)

    # Aggregate by year
    df_yearly = df.set_index("CET").resample("YE").mean(numeric_only=True)

    plt.plot(
        df_yearly.index,
        df_yearly["Mean TemperatureC"],
        label="Mean Temperature",
        alpha=0.8,
        linewidth=2,
        marker="o",
    )
    plt.plot(
        df_yearly.index,
        df_yearly["Max TemperatureC"],
        label="Max Temperature",
        alpha=0.8,
        linewidth=2,
        marker="o",
    )
    plt.plot(
        df_yearly.index,
        df_yearly["Min TemperatureC"],
        label="Min Temperature",
        alpha=0.8,
        linewidth=2,
        marker="o",
    )
    plt.xlabel("Year")
    plt.ylabel("Temperature (°C)")
    plt.title("Temperature Trends Over Time (Yearly Average)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("output/temperature_trends.png", dpi=300, bbox_inches="tight")
    plt.close()

    # 2. Histogram of temperature distribution (Histogram - numeric data)
    plt.figure(figsize=fig_size)
    plt.hist(df["Mean TemperatureC"].dropna(), bins=50, alpha=0.7, edgecolor="black")
    plt.xlabel("Mean Temperature (°C)")
    plt.ylabel("Frequency")
    plt.title("Distribution of Mean Daily Temperatures")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("output/temperature_histogram.png", dpi=300, bbox_inches="tight")
    plt.close()

    # 3. Box plot of temperatures by weather events (Box plot - categorical x numeric)
    # Filter out rows with NaN events and get top events
    events_data = df.dropna(subset=["Events"])
    top_events = events_data["Events"].value_counts().head(8).index

    plt.figure(figsize=fig_size)

    temp_by_event = [
        events_data[events_data["Events"] == event]["Mean TemperatureC"].dropna()
        for event in top_events
    ]

    plt.boxplot(temp_by_event, tick_labels=top_events)
    plt.xlabel("Weather Events")
    plt.ylabel("Mean Temperature (°C)")
    plt.title("Temperature Distribution by Weather Events")
    plt.xticks(rotation=45, ha="right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("output/temperature_by_events.png", dpi=300, bbox_inches="tight")
    plt.close()

    # 4. 2D Histogram of temperature vs humidity (Heat map - numeric x numeric)
    # Better for large datasets than scatter plot
    plt.figure(figsize=fig_size)
    valid_temp_hum = df.dropna(subset=["Mean TemperatureC", "Mean Humidity"])
    plt.hist2d(
        valid_temp_hum["Mean TemperatureC"],
        valid_temp_hum["Mean Humidity"],
        bins=50,
        cmap="viridis",
    )
    plt.colorbar(label="Frequency")
    plt.xlabel("Mean Temperature (°C)")
    plt.ylabel("Mean Humidity (%)")
    plt.title("Relationship Between Temperature and Humidity")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("output/temp_humidity_scatter.png", dpi=300, bbox_inches="tight")
    plt.close()

    # 5. Bar chart of weather events frequency with log scale (Bar chart - categorical data)
    plt.figure(figsize=fig_size)
    event_counts = df["Events"].value_counts().head(10)
    plt.bar(range(len(event_counts)), event_counts.values)
    plt.xlabel("Weather Events")
    plt.ylabel("Frequency (log scale)")
    plt.title("Frequency of Weather Events")
    plt.xticks(range(len(event_counts)), event_counts.index, rotation=45, ha="right")
    plt.yscale("log")  # Use logarithmic scale for better visibility of small values
    plt.grid(True, alpha=0.3, which="both")
    plt.tight_layout()
    plt.savefig("output/weather_events_bar.png", dpi=300, bbox_inches="tight")
    plt.close()

    # 6. Line plot of precipitation over time - monthly aggregation (Line plot - numeric x numeric sequential)
    plt.figure(figsize=fig_size)

    # Filter to show only period with precipitation data and aggregate by month
    precip_data = df[df["Precipitationmm"] > 0]

    if len(precip_data) > 0:
        start_date = precip_data["CET"].min()
        end_date = precip_data["CET"].max()
        df_filtered = df[(df["CET"] >= start_date) & (df["CET"] <= end_date)]
        # Aggregate by month for better readability - sum of precipitation per month
        df_precip_monthly = (
            df_filtered.set_index("CET").resample("ME")["Precipitationmm"].sum()
        )
        plt.plot(
            df_precip_monthly.index,
            df_precip_monthly.values,
            alpha=0.8,
            linewidth=2,
            marker="o",
            markersize=4,
        )
    else:
        df_precip_monthly = df.set_index("CET").resample("ME")["Precipitationmm"].sum()
        plt.plot(
            df_precip_monthly.index, df_precip_monthly.values, alpha=0.8, linewidth=2
        )

    plt.xlabel("Date")
    plt.ylabel("Monthly Precipitation (mm)")
    plt.title("Precipitation Over Time (Monthly Total)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("output/precipitation_trends.png", dpi=300, bbox_inches="tight")
    plt.close()

    print("All visualizations have been generated successfully!")


visualize_data()

All visualizations have been generated successfully!
